In [1]:
import pandas as pd
import polars as pl
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append("../../../1_allele_collection/2_analysis")
from plot_utils import *

sys.path.append("../..")
from analyses_plot_utils import *

In [2]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"
meta_output = '../../../1_allele_collection/3_outputs'

In [3]:
orf_distances = pl.read_csv(f'{outputs_dir}/orf_distances.csv', null_values=".", infer_schema_length=10000)
df = pl.read_csv(f'{outputs_dir}/edges_df.csv', null_values=".", infer_schema_length=10000)


In [4]:
# dedup
df = df.with_columns(
    pl.when(pl.col("aa_change").is_null())
    .then(pl.lit("wt"))
    .otherwise(pl.col("aa_change"))
    .alias("aa_change")
)
df = df.with_columns(
    (pl.col("symbol") + "_" + 
     pl.col("ad_symbol") + "_" + 
     pl.col("aa_change")).alias("ppi_group_clean")
)

df = df.sort("db_mut_id", descending=True).unique(subset=["ppi_group_clean"], keep="first")

df

scoring_pla,scoring_pos,large_plate_name,retest_batch,condition,condition_name,assay_id,media_control,ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,db_source_plate,db_source_well,ad_source_plate,ad_source_well,per_well_integrated_intensity_gray,manual_score_growth,total_colony_area,Children_reExpandedColonies_Count,unique_id,manual_score_growth_updated,contamination,outlier_score,outlier_size_score,outlier_flag,size_score_raw,size_score,ad_symbol,symbol,aa_change,gene_variant,spdi,nt_change,chr_num,nuc_loc,ref_allele,…,g2p_Secondary structure (PDBe/SIFTS),g2p_Sequence conflict (UniProt),g2p_Sequence uncertainty (UniProt),g2p_Signal (UniProt),g2p_Site (UniProt),g2p_Substrate genes,g2p_Topological domain (UniProt),g2p_Transit peptide (UniProt),g2p_Transmembrane (UniProt),g2p_Ubiquitination,g2p_Zinc finger (UniProt),plddt_f32,rsa_f32,pLDDT_Category,RSA_Category,OMIM_IDs,inheritance_pattern,inheritance_source,disease_modules,disease_module_count,OMIM_disease_names,splicing_variant,am_pathogenicity,db_group,db_condition_group,ppi_group,edge_group,seq_ad,seq_db,db_seq_final_right,manual_correction_applied,consensus_score,aa_score,consensus_manual_score_growth,wt_consensus_score,log2fc,ppi_group_clean
str,str,str,str,str,str,i64,bool,i64,str,i64,i64,str,str,str,str,f64,f64,i64,i64,i64,f64,bool,bool,bool,bool,f64,f64,str,str,str,str,str,str,f64,f64,str,…,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,f64,str,bool,f64,str,str,str,str,i64,i64,str,bool,f64,f64,f64,f64,f64,str
"""VUSAPWT1B1r05_002""","""K09""","""VUSAPWT1B1r05_001_002_003_004_…","""VUSAPWT1B1""","""r11""","""LWH25""",1,false,100064217,null,3535,0,"""DisWTGDEh04""","""F02""","""CEGSint_Y8A_4""","""G06""",589.083688,4.0,1106,1,38119,4.0,false,false,false,false,651526.558522,606.635529,"""GRAMD2A""","""EMD""","""wt""",null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""3535.0_0.0_VUSAPWT1B1""","""3535.0_0.0_LWH25_VUSAPWT1B1""","""100064217.0_3535.0_0.0_VUSAPWT…","""100064217.0_3535.0_VUSAPWT1B1""",1,1,null,false,592.499542,5.056238,4.0,592.499542,0.0,"""EMD_GRAMD2A_wt"""
"""VUSAPWT1B1r05_033""","""C07""","""VUSAPWT1B1r05_031_032_033_034_…","""VUSAPWT1B1""","""r11""","""LWH25""",1,false,1611,null,2713,200868,"""VUSMutY8Dh018""","""H03""","""CEGSint_Y8A_6""","""D03""",467.966019,3.0,1108,1,298853,3.0,false,false,false,false,518506.348826,482.780585,"""SPERT""","""MLH1""","""Asp485Glu""","""MLH1_Asp485Glu""","""NC_000003.12:37028828:T:A""","""1455T>A""",3.0,3.7028829e7,"""T""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,"""NaN""","""NaN""",null,null,null,"""Rare genetic immune disease; R…",3.0,null,false,0.0747,"""2713.0_200868.0_VUSAPWT1B1""","""2713.0_200868.0_LWH25_VUSAPWT1…","""1611.0_2713.0_200868.0_VUSAPWT…","""1611.0_2713.0_VUSAPWT1B1""",1,1,"""Y""",false,376.416905,9.465173,3.0,395.512624,-0.071208,"""MLH1_SPERT_Asp485Glu"""
"""VUSAPWT6B1r05_025""","""M07""","""VUSAPWT6B1r05_025_026_027_xxx_…","""VUSAPWT6B1""","""r17""","""LWH25""",6,false,14387,null,7652,1824,"""VUSMutY8Dh008""","""F10""","""CEGSint_Y8A6_2""","""A04""",111.111112,0.0,262,2,253733,0.0,false,false,false,false,29111.111445,27.105318,"""CENPP""","""STXBP1""","""Leu365Phe""","""STXBP1_Leu365Phe""","""NC_000009.12:127673243:C:T""","""1093C>T""",9.0,1.27673244e8,"""C""",…,null,null,null,null,null,null,null,null,null,null,null,97.9,0.0,""">90""","""<10""",null,null,null,"""Rare genetic developmental def…",5.0,null,false,0.9892,"""7652.0_1824.0_VUSAPWT6B1""","""7652.0_1824.0_LWH25_VUSAPWT6B1""","""14387.0_7652.0_1824.0_VUSAPWT6…","""14387.0_7652.0_VUSAPWT6B1""",1,1,"""Y""",false,13.552659,9.056894,0.0,595.752226,-5.357777,"""STXBP1_CENPP_Leu365Phe"""
"""VUSAPWT2B1r04_019""","""E13""","""VUSAPWT2B1r04_019_020_021_022_…","""VUSAPWT2B1""","""r16""","""LWH10""",2,false,6490,null,2920,0,"""DisWTY8Dh03""","""A03""","""CEGSint_Y8A2_2""","""H04""",500.990869,2.0,939,1,184523,2.

In [5]:
# keep only relevant columns for this notebook

# keep only relevant columns for this notebook
df[[
    'db_mut_id', 
    'symbol', 
    'ad_symbol', 
    'aa_change',
    'log2fc',
    'clinvar_clnsig_clean_pp_strict'
]].write_csv('./tmp/edges_df_luke.csv')

In [16]:
# Tables for Leo

# df.select(['symbol', 'db_orf_id', 'aa_change', 'db_mut_id', 'ad_symbol', 'ad_orf_id', 'clinvar_clnsig_clean', 'consensus_score', 'wt_consensus_score', 'log2fc']).write_csv(f"{outputs_dir}/ppi_level_scores_minimal_Leo.csv")

# orf_distances[['symbol', 'db_orf_id', 'aa_change', 'db_mut_id', 'gene_variant' ,'clinvar_clnsig_clean', 'norm_dist_rms']].write_csv(f"{outputs_dir}/orf_distances_minimal_Leo.csv")

In [6]:
orf_distances.filter(
    pl.col("clinvar_clnsig_clean_pp_strict").is_in(["Pathogenic", "Benign", "Likely pathogenic", "Likely benign"])
).with_columns(
    pl.when(
        pl.col("clinvar_clnsig_clean_pp_strict").is_in(["Pathogenic", "Likely pathogenic"])
    )
    .then(pl.lit("Positive"))
    .otherwise(pl.lit("Negative"))
    .alias("referenceSet"),
    pl.col("norm_dist_rms").alias("score"),
    pl.col("gene_variant").alias("hgvsp")
).select(
    pl.col(["hgvsp", "score", "referenceSet", "norm_dist_rms", "gene_variant", "clinvar_clnsig_clean_pp_strict"])
).write_csv("../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv")

orf_distances.with_columns(
    pl.col("norm_dist_rms").alias("score"),
    pl.col("gene_variant").alias("hgvs_pro")
).select(pl.col(["hgvs_pro", "score"])).write_csv("../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv")

In [7]:
%%bash
source /opt/homebrew/Caskroom/miniconda/base/etc/profile.d/conda.sh
conda activate py310

Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_bcv" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_bcv.log" --bandwidth "bcv" --printTransitions  --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_nrd0" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_nrd0.log" --bandwidth "nrd0" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_default" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_default.log" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_nrd" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_nrd.log" --bandwidth "nrd" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_sj" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_sj.log" --printTransitions --bandwidth "SJ" --kernel "gaussian" --outlierSuppression 0.1

Loading required package: parallel
Loading required package: maveLLR

Attaching package: ‘maveLLR’

The following objects are masked _by_ ‘.GlobalEnv’:

    buildLLR.gauss, buildLLR.kernel, drawDensityLLR

Warning message:
NAs introduced by coercion 


2026-01-16_16:21:03 INFO: Reading input 
2026-01-16_16:21:03 INFO: Calculating densties 


Warning message:
In stats::bw.bcv(data) : minimum occurred at one end of the range


2026-01-16_16:21:04 INFO: Drawing LLR plot 
2026-01-16_16:21:04 INFO: Drawing PRC curve 
  |======================================================================| 100%2026-01-16_16:21:05 INFO: Category transitions:  
0.542 : none -> patho.support
0.643 : patho.support -> patho.moderate
0.715 : patho.moderate -> patho.support
2026-01-16_16:21:05 INFO: Bootstrapping confidence intervals at  10000  iterations 
2026-01-16_16:21:05 INFO: Writing output to file  ../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_bcv.csv 
2026-01-16_16:21:05 INFO: Done! 


In [8]:
LLR_threshold = 0.542

# add ppi_hit column

orf_distances = orf_distances.with_columns(
    pl.when(pl.col("norm_dist_rms") >= LLR_threshold)
    .then(True)
    .otherwise(False)
    .alias("ppi_hit")
)


cols_to_keep = ['db_orf_id',
 'db_mut_id',
 'num_edges',
 'clinvar_clnsig_clean',
 'norm_dist_rms',
 'symbol',
 'gene_variant',
 'aa_change',
 'clinvar_clnsig_clean_pp_strict',
 'ppi_hit'
 ]

orf_distances = orf_distances.select(cols_to_keep)

orf_distances.write_csv(f'{outputs_dir}/orf_distances_final.csv')